# Ablation Evaluation — NLG + Grounding metrics per checkpoint

Loops over every checkpoint in `ablation_study/weights/<config>/`, rebuilds the
projector and LoRA-LLM with the CORRECT `num_queries` / `lora_r` for that config
(parsed from the folder name), generates reports, and computes:
- **NLG:** BLEU-1..4, ROUGE-L, METEOR, BERTScore
- **Grounding:** mean heatmap pairwise similarity (collapse check)

Then assembles one final ablation table.



## Cell 1 — Install + mount

In [ ]:
!pip install -q -U bitsandbytes transformers peft accelerate datasets evaluate rouge_score nltk bert_score
from google.colab import drive
drive.mount('/content/drive')
import nltk
nltk.download('punkt'); nltk.download('punkt_tab'); nltk.download('wordnet')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 22.3 MB/s eta 0:00:00
Mounted at /content/drive


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

## Cell 2 — Imports, device, paths

In [ ]:
import os, re, gc, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm
from datasets import load_dataset
from transformers import (AutoImageProcessor, AutoModel, AutoTokenizer,
                          AutoModelForCausalLM, BitsAndBytesConfig)
from peft import PeftModel, LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

ABLATION_ROOT = "/content/drive/MyDrive/ablation_study"
WEIGHTS_ROOT  = os.path.join(ABLATION_ROOT, "weights")
EVAL_CSV      = os.path.join(ABLATION_ROOT, "ablation_eval_metrics.csv")
SEED = 42
N_SAMPLE_HEATMAP = 150   # images used for the heatmap-variance collapse check

Device: cuda


## Cell 3 — Load base models ONCE (vision encoder, tokenizer, base LLM)

In [ ]:
print("Loading Rad-DINO + tokenizer...")
vision_processor = AutoImageProcessor.from_pretrained("microsoft/rad-dino")
vision_encoder = AutoModel.from_pretrained("microsoft/rad-dino").to(device)
vision_encoder.eval()

LLM_ID = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                         bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
print("Loading base Qwen (4-bit) once...")
base_llm = AutoModelForCausalLM.from_pretrained(LLM_ID, quantization_config=bnb,
                                                device_map={"": 0})
LLM_HIDDEN = base_llm.config.hidden_size
print("Base models ready. LLM hidden size:", LLM_HIDDEN)

Loading Rad-DINO + tokenizer...


preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base models ready. LLM hidden size: 2048


## Cell 4 — Holdout test set (same split/seed as training)

In [ ]:
print("Loading test split...")
full = load_dataset("Shrey-1329/cxiu_hf_dataset", split="train")
tt = full.train_test_split(test_size=0.2, seed=SEED)
vt = tt['test'].train_test_split(test_size=0.5, seed=SEED)
test_split = vt['test']   # untouched 10%
print("Test images:", len(test_split))

Loading test split...


README.md:   0%|          | 0.00/1.87k [00:00<?, ?B/s]

data/train-00000-of-00003-34bea7eff0b2e4(…):   0%|          | 0.00/369M [00:00<?, ?B/s]

data/train-00001-of-00003-e264f8a8545640(…):   0%|          | 0.00/369M [00:00<?, ?B/s]

data/train-00002-of-00003-b727c3f3dd3884(…):   0%|          | 0.00/370M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6060 [00:00<?, ? examples/s]

Test images: 606


## Cell 5 — Projector definition (with return_attention for grounding)

In [ ]:
class QFormerProjector(nn.Module):
    def __init__(self, num_queries, llm_dim, vision_dim=768):
        super().__init__()
        self.query_tokens = nn.Parameter(torch.randn(1, num_queries, vision_dim))
        self.cross_attn = nn.MultiheadAttention(vision_dim, num_heads=8, batch_first=True)
        self.norm1 = nn.LayerNorm(vision_dim)
        self.ffn = nn.Sequential(nn.Linear(vision_dim, vision_dim*4), nn.GELU(),
                                 nn.Linear(vision_dim*4, vision_dim))
        self.norm2 = nn.LayerNorm(vision_dim)
        self.proj = nn.Linear(vision_dim, llm_dim)
        self.llm_norm = nn.LayerNorm(llm_dim)
    def forward(self, patch_embeddings, return_attention=False):
        sp = patch_embeddings[:, 1:, :]
        q = self.query_tokens.expand(sp.shape[0], -1, -1)
        a, attn = self.cross_attn(q, sp, sp, need_weights=return_attention,
                                  average_attn_weights=True)
        x = self.norm1(q + a)
        x = self.norm2(x + self.ffn(x))
        out = self.llm_norm(self.proj(x))
        if return_attention:
            return out, attn
        return out

## Cell 6 — Helpers: parse config name, load a checkpoint



In [ ]:
def parse_config_name(name):
    """K32_r16 -> (32, 16)"""
    k = int(re.search(r"K(\d+)", name).group(1))
    r = int(re.search(r"r(\d+)", name).group(1))
    return k, r

def load_checkpoint(config_name):
    k, r = parse_config_name(config_name)
    ckpt_dir = os.path.join(WEIGHTS_ROOT, config_name)

    # rebuild projector at correct K
    projector = QFormerProjector(num_queries=k, llm_dim=LLM_HIDDEN).to(device)
    projector.load_state_dict(
        torch.load(os.path.join(ckpt_dir, "vision_projector.pth"),
                   map_location=device, weights_only=True))
    projector.eval()

    # attach the LoRA adapters for this config to the shared base LLM
    llm = PeftModel.from_pretrained(base_llm,
            os.path.join(ckpt_dir, "qwen_lora_adapters"))
    llm.eval()
    return projector, llm, k, r

## Cell 7 — Generation for one checkpoint

In [ ]:
EVAL_PROMPT = ("<|im_start|>system\n"
    "You are an expert radiologist. Write ONLY the diagnostic narrative.<|im_end|>\n"
    "<|im_start|>user\nAnalyze this chest radiograph and generate a concise clinical report.<|im_end|>\n"
    "<|im_start|>assistant\n")

def generate_reports(projector, llm):
    prompt_ids = tokenizer(EVAL_PROMPT, add_special_tokens=False,
                           return_tensors="pt").input_ids.to(device)
    gens, refs = [], []
    with torch.no_grad():
        for item in tqdm(test_split, desc="generating", leave=False):
            image = item['image'].convert('RGB')
            pv = vision_processor(images=image, return_tensors="pt").pixel_values.to(device)
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                pe = vision_encoder(pv).last_hidden_state.to(torch.float16)
                vprompt = projector(pe)
                pe_txt = llm.get_input_embeddings()(prompt_ids)
                emb = torch.cat([vprompt, pe_txt], dim=1)
                am = torch.ones(emb.shape[:2], dtype=torch.long, device=device)
                out = llm.generate(inputs_embeds=emb, attention_mask=am,
                                   max_new_tokens=120, min_new_tokens=20,
                                   num_beams=4, do_sample=False, length_penalty=1.2,
                                   repetition_penalty=1.1, early_stopping=True,
                                   pad_token_id=tokenizer.pad_token_id,
                                   eos_token_id=tokenizer.eos_token_id)
            gen = tokenizer.decode(out[0], skip_special_tokens=True).strip()
            ref = item.get('text', None)
            if ref is None or not ref.strip():
                continue
            gens.append(gen); refs.append(ref.strip())
    return gens, refs

## Cell 8 — NLG metrics (nltk BLEU 1-4, ROUGE-L, METEOR, BERTScore)

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

_smooth = SmoothingFunction().method1
_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
_bw = {"BLEU-1":(1,0,0,0), "BLEU-2":(0.5,0.5,0,0),
       "BLEU-3":(1/3,1/3,1/3,0), "BLEU-4":(0.25,0.25,0.25,0.25)}

def compute_nlg(gens, refs):
    bl = {k: [] for k in _bw}; rl = []; me = []
    for g, r in zip(gens, refs):
        gt, rt = g.lower().split(), r.lower().split()
        for name, w in _bw.items():
            bl[name].append(sentence_bleu([rt], gt, weights=w, smoothing_function=_smooth))
        rl.append(_rouge.score(r, g)['rougeL'].fmeasure)
        me.append(meteor_score([rt], gt))
    P, R, F1 = bert_score_fn(gens, refs, lang="en", verbose=False)
    out = {k: float(np.mean(v)) for k, v in bl.items()}
    out["ROUGE-L"] = float(np.mean(rl))
    out["METEOR"] = float(np.mean(me))
    out["BERTScore-F1"] = float(F1.mean().item())
    # uniqueness / collapse stat
    out["unique_reports"] = len(set(g.strip() for g in gens))
    out["total_reports"] = len(gens)
    out["pct_unique"] = round(100*out["unique_reports"]/max(out["total_reports"],1), 1)
    return out

## Cell 9 — Heatmap-variance (grounding collapse check) per config

In [ ]:
def compute_heatmap_variance(projector, n_sample=N_SAMPLE_HEATMAP):
    idxs = list(range(min(n_sample, len(test_split))))
    maps = []
    with torch.no_grad():
        for i in tqdm(idxs, desc="heatmaps", leave=False):
            image = test_split[i]['image'].convert('RGB')
            pv = vision_processor(images=image, return_tensors="pt").pixel_values.to(device)
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                pe = vision_encoder(pv).last_hidden_state.to(torch.float16)
                _, attn = projector(pe, return_attention=True)
            vec = attn.squeeze(0).mean(dim=0).float().cpu().numpy()
            lo, hi = vec.min(), vec.max()
            if hi - lo > 1e-8: vec = (vec - lo)/(hi - lo)
            maps.append(vec)
    H = np.stack(maps, 0)
    norms = np.linalg.norm(H, axis=1, keepdims=True); norms[norms==0] = 1e-8
    Hn = H / norms
    rng = np.random.default_rng(SEED); n = Hn.shape[0]
    ii = rng.integers(0, n, 2000); jj = rng.integers(0, n, 2000); m = ii != jj
    cos = np.sum(Hn[ii[m]] * Hn[jj[m]], axis=1)
    return float(cos.mean())

## Cell 11 — Run evaluation over ALL checkpoints

Resume-safe: skips configs already in the eval CSV. Frees the PEFT adapters
between configs so VRAM doesn't accumulate.

In [ ]:
config_names = sorted([d for d in os.listdir(WEIGHTS_ROOT)
                       if os.path.isdir(os.path.join(WEIGHTS_ROOT, d))])
print("Found configs:", config_names)

rows = []
if os.path.exists(EVAL_CSV):
    prev = pd.read_csv(EVAL_CSV); rows = prev.to_dict("records")
    done = set(prev["config"]); print("Already evaluated:", done)
else:
    done = set()

for name in config_names:
    if name in done:
        print(f"skip {name}"); continue
    print(f"\n===== EVALUATING {name} =====")
    projector, llm, k, r = load_checkpoint(name)

    gens, refs = generate_reports(projector, llm)
    nlg = compute_nlg(gens, refs)
    hmap_sim = compute_heatmap_variance(projector)
    pg, iou = compute_nih_localization(projector, llm)

    row = {"config": name, "num_queries": k, "lora_r": r,
           **{kk: round(vv, 4) if isinstance(vv, float) else vv
              for kk, vv in nlg.items()},
           "heatmap_pairwise_sim": round(hmap_sim, 4),
           "pointing_game": pg, "iou@0.5": iou}
    rows.append(row)
    pd.DataFrame(rows).to_csv(EVAL_CSV, index=False)
    print(f"  -> logged {name}")

    # free the per-config adapters + projector
    del projector, llm
    gc.collect(); torch.cuda.empty_cache()

print("\nDONE.")
final = pd.DataFrame(rows)
print(final)

Found configs: ['K16_r16', 'K32_r16', 'K64_r16', 'K64_r32', 'K64_r8']

===== EVALUATING K16_r16 =====


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(

generating: 100%|██████████| 606/606 [1:32:51<00:00,  9.02s/it]
                                                               

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  -> logged K16_r16

===== EVALUATING K32_r16 =====


generating:  33%|███▎      | 199/606 [29:42<1:01:49,  9.11s/it]

In [ ]:
# Evaluate ONLY the K32_r16 config
TARGET_CONFIG = "K32_r16"

# load existing results so we append, not overwrite
rows = []
if os.path.exists(EVAL_CSV):
    prev = pd.read_csv(EVAL_CSV)
    rows = prev.to_dict("records")
    done = set(prev["config"])
else:
    done = set()

if TARGET_CONFIG in done:
    print(f"{TARGET_CONFIG} already evaluated — skipping. Delete its row from the CSV to redo.")
else:
    print(f"\n===== EVALUATING {TARGET_CONFIG} =====")
    projector, llm, k, r = load_checkpoint(TARGET_CONFIG)

    gens, refs = generate_reports(projector, llm)
    nlg = compute_nlg(gens, refs)
    hmap_sim = compute_heatmap_variance(projector)
    pg, iou = compute_nih_localization(projector, llm)

    row = {"config": TARGET_CONFIG, "num_queries": k, "lora_r": r,
           **{kk: round(vv, 4) if isinstance(vv, float) else vv
              for kk, vv in nlg.items()},
           "heatmap_pairwise_sim": round(hmap_sim, 4),
           "pointing_game": pg, "iou@0.5": iou}
    rows.append(row)
    pd.DataFrame(rows).to_csv(EVAL_CSV, index=False)
    print(f"  -> logged {TARGET_CONFIG}")

    del projector, llm
    gc.collect(); torch.cuda.empty_cache()

print("\nCurrent results so far:")
print(pd.DataFrame(rows))


===== EVALUATING K32_r16 =====


generating:   0%|          | 0/606 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1119: UserWarning: Passing `repetition_penalty` with `inputs_embeds` and without `input_ids` to `generate` will apply the penalty only to newly generated tokens, not to the prompt.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  -> logged K32_r16

Current results so far:
    config  num_queries  lora_r  BLEU-1  BLEU-2  BLEU-3  BLEU-4  ROUGE-L  \
0  K16_r16           16      16  0.3345  0.1914  0.1184  0.0778   0.2815   
1  K32_r16           32      16  0.3540  0.2090  0.1332  0.0860   0.3000   

   METEOR  BERTScore-F1  unique_reports  total_reports  pct_unique  \
0  0.3115        0.8686               3            606         0.5   
1  0.3395        0.8713               6            606         1.0   

   heatmap_pairwise_sim  pointing_game  iou@0.5  
0                0.8189            NaN      NaN  
1                0.8730            NaN      NaN  


In [ ]:
# Evaluate ONLY the K64_r16 config (your main/baseline model)
TARGET_CONFIG = "K64_r16"

rows = []
if os.path.exists(EVAL_CSV):
    prev = pd.read_csv(EVAL_CSV)
    rows = prev.to_dict("records")
    done = set(prev["config"])
else:
    done = set()

if TARGET_CONFIG in done:
    print(f"{TARGET_CONFIG} already evaluated — skipping. Delete its row from the CSV to redo.")
else:
    print(f"\n===== EVALUATING {TARGET_CONFIG} =====")
    projector, llm, k, r = load_checkpoint(TARGET_CONFIG)

    gens, refs = generate_reports(projector, llm)
    nlg = compute_nlg(gens, refs)
    hmap_sim = compute_heatmap_variance(projector)
    pg, iou = compute_nih_localization(projector, llm)

    row = {"config": TARGET_CONFIG, "num_queries": k, "lora_r": r,
           **{kk: round(vv, 4) if isinstance(vv, float) else vv
              for kk, vv in nlg.items()},
           "heatmap_pairwise_sim": round(hmap_sim, 4),
           "pointing_game": pg, "iou@0.5": iou}
    rows.append(row)
    pd.DataFrame(rows).to_csv(EVAL_CSV, index=False)
    print(f"  -> logged {TARGET_CONFIG}")

    del projector, llm
    gc.collect(); torch.cuda.empty_cache()

print("\nAll results so far:")
print(pd.DataFrame(rows).to_string(index=False))


===== EVALUATING K64_r16 =====


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
generating:   0%|          | 0/606 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1119: UserWarning: Passing `repetition_penalty` with `inputs_embeds` and without `input_ids` to `generate` will apply the penalty only to newly generated tokens, not to the prompt.
  warnings.warn(


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  -> logged K64_r16

All results so far:
 config  num_queries  lora_r  BLEU-1  BLEU-2  BLEU-3  BLEU-4  ROUGE-L  METEOR  BERTScore-F1  unique_reports  total_reports  pct_unique  heatmap_pairwise_sim  pointing_game  iou@0.5
K16_r16           16      16  0.3345  0.1914  0.1184  0.0778   0.2815  0.3115        0.8686               3            606         0.5                0.8189            NaN      NaN
K32_r16           32      16  0.3540  0.2090  0.1332  0.0860   0.3000  0.3395        0.8713               6            606         1.0                0.8730            NaN      NaN
K64_r16           64      16  0.3330  0.1943  0.1233  0.0806   0.2905  0.3169        0.8706              17            606         2.8                0.8720            NaN      NaN


## Cell 12 — Assemble the final ablation table

In [ ]:
df = pd.read_csv(EVAL_CSV)
# order rows: K-ablation first (r=16), then rank-ablation (K=64)
df = df.sort_values(["lora_r", "num_queries"]).reset_index(drop=True)

cols = ["config","num_queries","lora_r","METEOR","BERTScore-F1",
        "ROUGE-L","BLEU-4","pct_unique","heatmap_pairwise_sim"]
cols = [c for c in cols if c in df.columns]
table = df[cols]
print("=== FINAL ABLATION TABLE ===")
print(table.to_string(index=False))

table.to_csv(os.path.join(ABLATION_ROOT, "final_ablation_table.csv"), index=False)
print("\nSaved final_ablation_table.csv")

=== FINAL ABLATION TABLE ===
 config  num_queries  lora_r  METEOR  BERTScore-F1  ROUGE-L  BLEU-4  pct_unique  heatmap_pairwise_sim
K16_r16           16      16  0.3115        0.8686   0.2815  0.0778         0.5                0.8189
K32_r16           32      16  0.3395        0.8713   0.3000  0.0860         1.0                0.8730
K64_r16           64      16  0.3169        0.8706   0.2905  0.0806         2.8                0.8720

Saved final_ablation_table.csv
